[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO Loss

Implement the **Group Relative Policy Optimization (GRPO)** loss — a group-wise, baseline-subtracted REINFORCE objective commonly used in RLAIF (reinforcement learning from AI feedback).

Given a batch of log-probabilities, scalar rewards, and group ids (one group per prompt), define the within-group normalized advantages:

$$A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}$$

where \(\bar r_{g(i)}\) and \(\text{std}_{g(i)}\) are the mean and standard deviation of rewards in the group of example \(i\).

The GRPO loss is then the negative advantage-weighted log-probability:

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i)\big].$$

### Signature
```python
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """GRPO loss over a batch.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
```

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import torch.nn.functional as F

In [46]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    ids, indices, counts = torch.unique(group_ids, return_inverse=True, return_counts=True)
    output = torch.scatter_reduce(ids.float(), dim=-1, index=indices, src=rewards, reduce="mean", include_self=False)
    r_bar = torch.gather(output, -1, indices)
    zero_mean_reward = rewards - r_bar
    output = torch.scatter_reduce(ids.float(), dim=-1, index=indices, src=zero_mean_reward*zero_mean_reward, reduce="sum", include_self=False) / counts
    r_var = torch.gather(output, -1, indices)
    advantage = zero_mean_reward / (torch.sqrt(r_var) + eps)
    return -(logps * advantage.detach()).mean()
    pass  # compute normalized advantages per group and return -mean(adv.detach() * logps)

In [47]:
# 🧪 Debug
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
# group_ids = torch.tensor([1, 0, 3, 0])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

Loss: -0.24997496604919434


In [48]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_loss')


🧪 Testing: GRPO (Group Relative Policy Optimization) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Basic shape & type (1.8ms)
  ✅ [2/4] Numeric check vs reference (1.9ms)
  ✅ [3/4] Gradient flows to logps only (9.8ms)
  ✅ [4/4] Group-wise normalization (3.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (17.3ms total)
  Progress saved. Run status() to see your dashboard.

